# Orion-XAI evaluation: does fine-tuning help?

Compares **base Llama 3.2 3B Instruct** with **Orion-XAI** (the same model + the LoRA adapter from Hugging Face).

**Kaggle settings before you run:**
1. Accelerator: **GPU T4 x1**
2. Internet: **On**
3. Optional, for the LLM judge: Add-ons → Secrets → add `GROQ_API_KEY` (free key from console.groq.com)

Runtime: about 5 minutes for multiple choice, 25–40 minutes with open-ended answers.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
# Uses the eval branch until its pull request is merged; then change it to main
!git clone -q --branch eval/base-vs-finetuned https://github.com/mohamedibrahim26/explainable-ai.git
%cd explainable-ai
!python eval/check_overlap.py | tail -3

In [ ]:
import os
RUN_JUDGE = False
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    RUN_JUDGE = True
    print("Groq key found: LLM judge will run.")
except Exception:
    print("No GROQ_API_KEY secret: skipping the LLM judge (multiple choice + generations still run).")

## Quick smoke test (3 items, ~2 minutes)
Run this first. If it works, run the full evaluation below.

In [ ]:
!python eval/run_eval.py --limit 3 --max-new-tokens 64 --out results_smoke

## Full evaluation

In [ ]:
judge_flag = "--judge" if RUN_JUDGE else ""
!python eval/run_eval.py --out results {judge_flag}

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/REPORT.md").read()))

## Save the results
Download `results/` from the Output panel (or the zip below) and commit it to the repo as `eval/results/`.
Then copy the tables from `REPORT.md` into the main README's **Evaluation** section.

In [ ]:
!zip -qr /kaggle/working/orion_eval_results.zip results
print("Saved /kaggle/working/orion_eval_results.zip")